# Dataset Generation

This combines everything learned from data_explore into a concise pipeline that joins our two sets of data

In [2]:
import pandas as pd

import warnings
import re

# set display options
# warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [3]:
# read in data
# spotify users playlists
spotify = pd.read_csv('../../data/spotify_dataset.csv', on_bad_lines='skip')
spotify.columns=["user","artist","track","playlist"]

# artist/track metadata
meta = pd.read_csv('../../data/spotify_dataset_2.csv')

In [4]:
# will merge by artist alone; create common field artist_clean
def clean_text(text):
    """
    Drops spaces and lower-cases text
    """
    return str(text).lower().replace(' ', '')

spotify['artist_clean'] = (spotify['artist'].apply(clean_text))
meta['artist_clean'] = (meta['Artist(s)'].apply(clean_text))

In [5]:
# convert datatypes in meta for artist aggregation
# convert length into seconds
def convert_to_seconds(time_str):
    minutes, seconds = map(int, time_str.split(':'))
    return minutes * 60 + seconds

meta['Length_Seconds'] = meta['Length'].apply(convert_to_seconds)

# convert Release Date to date
def convert_to_date(date_str):
    # Remove the ordinal suffix if present (e.g., 'st', 'nd', 'rd', 'th')
    date_str = re.sub(r'(?<=\d)(st|nd|rd|th)', '', date_str)
    return pd.to_datetime(date_str, format='%d %B %Y')

meta['Release_Date_Format'] = meta['Release Date'].apply(convert_to_date)

# drop db from Loudness (db) and make sure it is numerical
meta['loudness_db_num'] = meta['Loudness (db)'].str.replace('db', '').astype('float')

# drop song similarity (this is going to be artist level)
meta = meta.drop(['Similarity Score 1', 'Similarity Score 2', 'Similarity Score 3'], axis=1)

In [6]:
# pull all of the numerical columns into a list for averaging
numeric_cols = meta.select_dtypes(include=['number']).columns.tolist()

# take mean of all numerical columns
meta_artist_num = meta.groupby('artist_clean')[numeric_cols].mean().reset_index()
meta_artist_num.head()

,artist_clean,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num
0,!!!,118.000000,27.0625,83.312500,73.312500,71.125000,6.437500,18.187500,5.500000,11.6875,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0625,280.500000,-6.65875
1,"!!!,lealea",126.500000,33.5000,82.500000,82.000000,80.000000,6.000000,9.500000,2.500000,16.0000,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0000,275.500000,-6.39000
2,!marc¡,175.000000,0.0000,40.000000,72.000000,52.000000,28.000000,12.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,122.000000,-7.99000
3,"!yadnus,daylyt",82.333333,4.0000,57.333333,56.666667,88.666667,24.333333,29.666667,51.333333,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,81.666667,-10.52000
4,!zeesh,101.000000,51.0000,31.000000,64.000000,81.000000,38.000000,79.000000,85.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,112.000000,-13.78000


In [7]:
# group by categorical columns and keep top freq
categoric_cols = meta.select_dtypes(include=['object', 'datetime']).columns.tolist()
remove_list = ['Artist(s)', 'Release Date', 'song', 'text', 'Length', 
               'Album', 'Loudness (db)', 'Similar Song 1', 
               'Similar Song 2', 'Similar Song 3']

# Remove multiple specific items by their names using list comprehension
categoric_cols = [s for s in categoric_cols if s not in remove_list]
print(categoric_cols)

# for debugging purposes (remove once you find bad column)
categoric_cols = ['Genre', 'Key']

meta_artist_cat = meta.groupby('artist_clean')[categoric_cols].apply(lambda x: x.value_counts().index[0])
# reformat with tuple values in their own columns
meta_artist_cat = pd.DataFrame.from_records(meta_artist_cat.tolist(), columns=categoric_cols)
meta_artist_info = pd.merge(meta_artist_cat, meta_artist_num, left_index=True, right_index=True)
meta_artist_info.head()

['emotion', 'Genre', 'Key', 'Time signature', 'Explicit', 'Similar Artist 1', 'Similar Artist 2', 'Similar Artist 3', 'artist_clean', 'Release_Date_Format']


,Genre,Key,artist_clean,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num
0,hip hop,A Maj,!!!,118.000000,27.0625,83.312500,73.312500,71.125000,6.437500,18.187500,5.500000,11.6875,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0625,280.500000,-6.65875
1,hip hop,C Maj,"!!!,lealea",126.500000,33.5000,82.500000,82.000000,80.000000,6.000000,9.500000,2.500000,16.0000,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0000,275.500000,-6.39000
2,hip hop,C Maj,!marc¡,175.000000,0.0000,40.000000,72.000000,52.000000,28.000000,12.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,122.000000,-7.99000
3,hip hop,A Maj,"!yadnus,daylyt",82.333333,4.0000,57.333333,56.666667,88.666667,24.333333,29.666667,51.333333,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,81.666667,-10.52000
4,hip hop,D Maj,!zeesh,101.000000,51.0000,31.000000,64.000000,81.000000,38.000000,79.000000,85.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,112.000000,-13.78000


In [8]:
# make a table for converting artist/artist_clean values
artist_map = spotify[['artist_clean', 'artist']].drop_duplicates(subset='artist_clean')

# put in artist popularity metric
# number of users
num_users = spotify['user'].nunique()

# find most popular artists (count of users with artist in playlist)
# count artists first
artist_user_count = spotify[['user', 'artist_clean']].drop_duplicates() \
    .groupby('artist_clean')['user'].count()
artist_user_count = pd.DataFrame(artist_user_count).reset_index()
artist_user_count.columns = ['artist_clean', 'count']

# artist popularity
artist_user_count['spotify_popularity'] = artist_user_count['count'] / num_users
artist_user_count = artist_user_count[['artist_clean', 'spotify_popularity']]
artist_user_count.sort_values('spotify_popularity', ascending=False).head()

,artist_clean,spotify_popularity
49262,coldplay,0.291871
54528,daftpunk,0.291054
205962,rihanna,0.257067
58608,davidguetta,0.242556
38339,calvinharris,0.234452


In [9]:
# prep spotify for merging with artist_user_count and meta_artist_info
# group by user, artist, and playlist
spotify_artist = spotify.groupby(['user', 'artist_clean', 'playlist'])['track'].count().reset_index()
spotify_artist.columns = ['user', 'artist_clean', 'playlist', 'tracks']
spotify_artist.head()

,user,artist_clean,playlist,tracks
0,00055176fea33f6e027cd3302289378b,5secondsofsummer,favs,10
1,00055176fea33f6e027cd3302289378b,abigailbreslin,favs,1
2,00055176fea33f6e027cd3302289378b,againstthecurrent,favs,3
3,00055176fea33f6e027cd3302289378b,alltimelow,favs,8
4,00055176fea33f6e027cd3302289378b,auryn,favs,1


In [10]:
# merge all 3 artist datasets (spotify_artist, meta_artist_info, artist_user_count)
print(len(spotify_artist))
df = spotify_artist.merge(artist_user_count, how='left', on='artist_clean')
print(len(df))
df = df.merge(meta_artist_info, how='left', on='artist_clean')
print(len(df))
# merge in original artist names (artist_map) and drop artist_clean
df = df.merge(artist_map, how='left', on='artist_clean')
print(len(df))
df = df.drop('artist_clean', axis=1)
# move artist column to same spot where artist_clean was
df = df[['user', 'artist', 'playlist', 'tracks', 'spotify_popularity', 'Genre', 'Key',
       'Tempo', 'Popularity', 'Energy', 'Danceability', 'Positiveness',
       'Speechiness', 'Liveness', 'Acousticness', 'Instrumentalness',
       'Good for Party', 'Good for Work/Study',
       'Good for Relaxation/Meditation', 'Good for Exercise',
       'Good for Running', 'Good for Yoga/Stretching', 'Good for Driving',
       'Good for Social Gatherings', 'Good for Morning Routine',
       'Length_Seconds', 'loudness_db_num']]
# check work
df.head()

4353512
4353512
4353512
4353512


,user,artist,playlist,tracks,spotify_popularity,Genre,Key,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num
0,00055176fea33f6e027cd3302289378b,5 Seconds Of Summer,favs,10,0.025129,"pop,pop rock,pop punk",D Maj,134.162338,44.811688,80.545455,53.415584,45.896104,11.272727,27.863636,5.409091,0.655844,0.175325,0.000000,0.0,0.285714,0.162338,0.0,0.038961,0.006494,0.032468,210.675325,-4.913896
1,00055176fea33f6e027cd3302289378b,Abigail Breslin,favs,1,0.000440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00055176fea33f6e027cd3302289378b,Against The Current,favs,3,0.001571,"acoustic,pop rock,rock",G Maj,121.264706,37.147059,84.441176,58.529412,57.176471,5.411765,16.058824,4.205882,0.000000,0.088235,0.000000,0.0,0.352941,0.088235,0.0,0.029412,0.000000,0.029412,199.794118,-4.070000
3,00055176fea33f6e027cd3302289378b,All Time Low,favs,8,0.030531,"alternative rock,emo,pop punk",D Maj,142.752066,38.801653,87.652893,48.074380,53.223140,7.842975,20.074380,3.694215,0.528926,0.082645,0.008264,0.0,0.206612,0.049587,0.0,0.000000,0.000000,0.000000,202.413223,-4.211901
4,00055176fea33f6e027cd3302289378b,Auryn,favs,1,0.008355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# quantify metadata that did not match up to explain row drops
original_size = len(df)
print(f'Rows before dropping null: {original_size}')
df = df.dropna()
new_size = len(df)
print(f'Rows after dropping null: {new_size}')
# calculate %
print(f'Percentage of data remaining: {round(new_size / original_size, 2) * 100:.0f}%')

Rows before dropping null: 4353512
Rows after dropping null: 3027724
Percentage of data remaining: 70%


In [12]:
# fix column labels (lowercase, underscores)
df.columns = [col.lower().replace(' ', '_').replace('/', '_') for col in df.columns]
df

,user,artist,playlist,tracks,spotify_popularity,genre,key,tempo,popularity,energy,danceability,positiveness,speechiness,liveness,acousticness,instrumentalness,good_for_party,good_for_work_study,good_for_relaxation_meditation,good_for_exercise,good_for_running,good_for_yoga_stretching,good_for_driving,good_for_social_gatherings,good_for_morning_routine,length_seconds,loudness_db_num
0,00055176fea33f6e027cd3302289378b,5 Seconds Of Summer,favs,10,0.025129,"pop,pop rock,pop punk",D Maj,134.162338,44.811688,80.545455,53.415584,45.896104,11.272727,27.863636,5.409091,0.655844,0.175325,0.000000,0.000000,0.285714,0.162338,0.000000,0.038961,0.006494,0.032468,210.675325,-4.913896
2,00055176fea33f6e027cd3302289378b,Against The Current,favs,3,0.001571,"acoustic,pop rock,rock",G Maj,121.264706,37.147059,84.441176,58.529412,57.176471,5.411765,16.058824,4.205882,0.000000,0.088235,0.000000,0.000000,0.352941,0.088235,0.000000,0.029412,0.000000,0.029412,199.794118,-4.070000
3,00055176fea33f6e027cd3302289378b,All Time Low,favs,8,0.030531,"alternative rock,emo,pop punk",D Maj,142.752066,38.801653,87.652893,48.074380,53.223140,7.842975,20.074380,3.694215,0.528926,0.082645,0.008264,0.000000,0.206612,0.049587,0.000000,0.000000,0.000000,0.000000,202.413223,-4.211901
5,00055176fea33f6e027cd3302289378b,Austin Mahone,favs,1,0.016145,pop,B Maj,111.692308,31.871795,63.461538,67.384615,56.435897,5.846154,15.769231,18.256410,0.000000,0.153846,0.076923,0.051282,0.230769,0.025641,0.000000,0.128205,0.000000,0.230769,198.230769,-5.960256
6,00055176fea33f6e027cd3302289378b,Avril Lavigne,favs,2,0.071806,"rock,pop,alternative rock",F Maj,128.726316,57.505263,79.252632,52.368421,48.052632,5.836842,23.073684,6.663158,0.363158,0.157895,0.005263,0.000000,0.242105,0.152632,0.000000,0.005263,0.010526,0.021053,214.731579,-4.756263
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4353507,fff77dadf8528083c920b9c018847e8b,Walk the Moon,Liked from Radio,1,0.062194,"indie rock,pop rock",C Maj,125.352941,38.450980,76.803922,57.921569,53.333333,5.705882,17.980392,6.039216,1.176471,0.156863,0.019608,0.019608,0.450980,0.098039,0.000000,0.000000,0.000000,0.058824,237.352941,-5.650784
4353508,fff77dadf8528083c920b9c018847e8b,We Are Scientists,Liked from Radio,1,0.027956,indie rock,F# min,135.500000,23.562500,90.625000,44.625000,49.250000,8.000000,20.687500,1.875000,4.750000,0.000000,0.000000,0.000000,0.125000,0.000000,0.000000,0.000000,0.000000,0.000000,186.875000,-4.510625
4353509,fff77dadf8528083c920b9c018847e8b,Wye Oak,Liked from Radio,1,0.025694,"indie rock,dream pop",C Maj,128.151515,21.757576,59.727273,48.848485,34.545455,3.818182,21.515152,27.666667,28.000000,0.000000,0.060606,0.030303,0.090909,0.030303,0.000000,0.030303,0.000000,0.030303,242.848485,-8.526061
4353510,fff77dadf8528083c920b9c018847e8b,Yeah Yeah Yeahs,Liked from Radio,1,0.123634,"rock,alternative rock,post-punk",A Maj,131.178571,32.714286,83.375000,46.214286,36.071429,9.196429,27.357143,9.750000,19.232143,0.053571,0.035714,0.017857,0.196429,0.035714,0.017857,0.000000,0.000000,0.000000,213.964286,-5.616786


In [ ]:
# output data to file for easy import
df.to_csv('../../data/clean_data.csv', index=False)